# Inspect data-quality content

Read-only inventory of everything this module creates or writes in the CDF project:

1. **Records stream** (`settings.yaml` → `records.stream_id`) — `DataQualityValidationRecord` and `RuleEngineResult`
2. **Catalog** — DataProducts and RuleSets published by Toolkit
3. **Runtime** — Function, workflows, triggers, monitor schedules
4. **DMS state** — spaces, containers/views, historic queue jobs in `dataQualitySpace`

Credentials come from the Toolkit `.env` next to `cdf.toml` (`CDF_*` / `IDP_*`), same as `deploy_infrastructure.py` and `manage_rulesets.ipynb`.

The records stream created by this module is **immutable** (`BasicArchive`). Filter/aggregate calls therefore require a `lastUpdatedTime` window (see `DAYS` in the first code cell).

Edit all tunables in the **first code cell** (`DAYS`, record limits, `DUMP_FULL_LOGS`, …).

In [ ]:
import sys
from datetime import UTC, datetime, timedelta
from pathlib import Path

SCRIPTS_DIR = next(
    path for path in (Path.cwd(), Path.cwd() / "modules/common/cdf_dq_runtime/scripts") if (path / "_cli.py").is_file()
)
sys.path.insert(0, str(SCRIPTS_DIR))

from _cli import (  # noqa: E402
    configure_deploy_cli,
    create_client_from_env,
    find_toolkit_project_root,
    load_project_env,
    load_variables,
)

configure_deploy_cli()
print(f"Credentials: {load_project_env()}")
client = create_client_from_env()
API_URL = f"/api/v1/projects/{client.config.project}"
RULESETS_URL = f"{API_URL}/rulesets"
DATA_PRODUCTS_URL = f"{API_URL}/dataproducts"
ALPHA_HEADERS = {"cdf-version": "alpha"}

_project_root = find_toolkit_project_root()
_toolkit_config = None
if _project_root is not None:
    for _name in ("config.dev.yaml", "config.local.yaml", "config.prod.yaml"):
        _candidate = _project_root / _name
        if _candidate.is_file():
            _toolkit_config = _candidate
            break

_vars = load_variables(_toolkit_config)

# --- Tunable options (edit these) -------------------------------------------------
DAYS = 7  # Records look-back window (immutable streams require lastUpdatedTime)
MAX_VALIDATION_RECORDS = 20  # Top N DataQualityValidationRecord rows to list
MAX_RULE_ENGINE_RECORDS = 20  # Top N RuleEngineResult rows to list
RECENT_FUNCTION_CALLS_LIMIT = 15  # Calls listed in list_recent_function_calls()
MODE_INSPECT_LIMIT = 40  # Calls scanned by inspect_function_modes()
MAX_ERROR_SAMPLES = 5  # Distinct error lines printed per mode
DUMP_FULL_LOGS = False  # True → print full logs for problem modes
FULL_LOG_MAX_LINES = None  # None = entire log; else last N lines per dump
MAX_FULL_LOG_CALLS = 2  # Max problem calls to dump per mode

# --- Module / CDF identifiers -----------------------------------------------------
FUNCTION_EXTERNAL_ID = "data-quality-validation"
VALIDATION_CONTAINER = "DataQualityValidationRecord"
RULE_ENGINE_CONTAINER = "RuleEngineResult"
DQ_WORKFLOW_PREFIXES = ("dq-shacl-", "dq-ts-shacl-", "dq-data-product-sync", "dq-historic-queue-manager")
HISTORIC_JOB_QUEUE_VIEW = "HistoricJobQueueView"
HISTORIC_JOB_QUEUE_VIEW_VERSION = "1"

# --- From Toolkit config / defaults -----------------------------------------------
DATA_QUALITY_SPACE = _vars.get("dataQualitySpace") or "dataQuality"
RECORDS_STREAM_ID = _vars.get("recordsStreamId") or "test_stream_1"
DATA_PRODUCT_EXTERNAL_ID = _vars.get("dataProductExternalId") or ""
DATA_PRODUCT_SCHEMA_SPACE = _vars.get("dataProductSchemaSpace") or ""

print(f"Project: {client.config.project}")
print(f"dataQualitySpace: {DATA_QUALITY_SPACE}")
print(f"recordsStreamId: {RECORDS_STREAM_ID}")
print(f"dataProductExternalId: {DATA_PRODUCT_EXTERNAL_ID}")
print(f"dataProductSchemaSpace: {DATA_PRODUCT_SCHEMA_SPACE}")
print(
    f"Options: DAYS={DAYS}, MAX_VALIDATION_RECORDS={MAX_VALIDATION_RECORDS}, "
    f"DUMP_FULL_LOGS={DUMP_FULL_LOGS}"
)

In [ ]:
def _list_items(url: str, limit: int = 100) -> list[dict]:
    items: list[dict] = []
    params: dict[str, object] = {"limit": limit}
    while True:
        body = client.get(url, params=params, headers=ALPHA_HEADERS).json()
        items.extend(body.get("items", []))
        if not body.get("nextCursor"):
            return items
        params["cursor"] = body["nextCursor"]


def _ms_ago(days: float) -> int:
    return int((datetime.now(UTC) - timedelta(days=days)).timestamp() * 1000)


def _record_props(record: dict, container: str, space: str | None = None) -> dict:
    resolved_space = space or DATA_QUALITY_SPACE
    return record.get("properties", {}).get(resolved_space, {}).get(container, {}) or {}


def _with_alpha_headers():
    """Context-style helper: set alpha header for Records/Streams, restore afterwards."""

    class _Alpha:
        def __enter__(self):
            self._original = dict(client.config.headers)
            client.config.headers["cdf-version"] = "alpha"
            return client

        def __exit__(self, exc_type, exc, tb) -> None:
            client.config.headers.clear()
            client.config.headers.update(self._original)

    return _Alpha()


def get_stream(stream_id: str | None = None) -> dict:
    """Return stream metadata (type, template, limits)."""
    resolved = stream_id or RECORDS_STREAM_ID
    with _with_alpha_headers():
        return client.get(f"{API_URL}/streams/{resolved}").json()


def filter_records(
    *,
    container: str = VALIDATION_CONTAINER,
    stream_id: str | None = None,
    days: float = 7,
    limit: int = 100,
    max_records: int | None = 500,
    equals: dict[str, object] | None = None,
    space: str | None = None,
) -> list[dict]:
    """Filter records from the configured stream (paginated).

    Args:
        container: Record container external ID.
        stream_id: Stream external ID (default from module config).
        days: Look-back window for ``lastUpdatedTime`` (required on immutable streams).
        limit: Page size for each API call.
        max_records: Stop after this many records (``None`` = all pages in the window).
        equals: Optional property → value equals filters on the container.
        space: Records space (default ``dataQualitySpace``).
    """
    resolved_stream = stream_id or RECORDS_STREAM_ID
    resolved_space = space or DATA_QUALITY_SPACE
    body: dict[str, object] = {
        "lastUpdatedTime": {"gt": _ms_ago(days)},
        "limit": limit,
    }
    if equals:
        if len(equals) == 1:
            key, value = next(iter(equals.items()))
            body["filter"] = {"equals": {"property": [resolved_space, container, key], "value": value}}
        else:
            body["filter"] = {
                "and": [
                    {"equals": {"property": [resolved_space, container, key], "value": value}}
                    for key, value in equals.items()
                ]
            }

    items: list[dict] = []
    with _with_alpha_headers():
        while True:
            response = client.post(f"{API_URL}/streams/{resolved_stream}/records/filter", json=body).json()
            page = response.get("items", [])
            for record in page:
                props = (record.get("properties") or {}).get(resolved_space, {})
                if container not in props:
                    continue
                items.append(record)
                if max_records is not None and len(items) >= max_records:
                    return items
            cursor = response.get("nextCursor")
            if not cursor:
                return items
            body["cursor"] = cursor
    return items


def aggregate_records(
    *,
    container: str = VALIDATION_CONTAINER,
    stream_id: str | None = None,
    days: float = 7,
    space: str | None = None,
) -> dict:
    """Aggregate validation/inference counts for the look-back window."""
    resolved_stream = stream_id or RECORDS_STREAM_ID
    resolved_space = space or DATA_QUALITY_SPACE
    payload: dict[str, object] = {
        "lastUpdatedTime": {"gt": _ms_ago(days)},
        "aggregates": {
            "totalCount": {"count": {}},
            "ruleSetIds": {"uniqueValues": {"property": [resolved_space, container, "ruleSetId"]}},
            "passedBreakdown": {"uniqueValues": {"property": [resolved_space, container, "passedValidation"]}},
            "jobRuns": {"uniqueValues": {"property": [resolved_space, container, "jobRunId"]}},
        },
    }
    if container == VALIDATION_CONTAINER:
        # Scope count to validation rows when the stream also holds RuleEngineResult.
        payload["filter"] = {
            "exists": {"property": [resolved_space, container, "passedValidation"]}
        }
    with _with_alpha_headers():
        return client.post(f"{API_URL}/streams/{resolved_stream}/records/aggregate", json=payload).json()

## Records stream

Shows stream metadata, aggregate counts, then sample validation / rule-engine records.
Tune `DAYS`, `MAX_VALIDATION_RECORDS`, and `MAX_RULE_ENGINE_RECORDS` in the first code cell.

Function logs for `data-quality-validation` look like `Posted [PASSED] record for …` / `Posted [FAILED] record for …` when writes succeed.

In [ ]:
stream = get_stream()
print(f"Stream: {stream.get('externalId')}")
print(f"  type: {stream.get('type')} (template: {stream.get('createdFromTemplate')})")
print(f"  createdTime: {stream.get('createdTime')}")
limits = (stream.get("settings") or {}).get("limits") or {}
print(f"  maxFilteringInterval: {limits.get('maxFilteringInterval')}")
print(f"  maxRecordsTotal: {limits.get('maxRecordsTotal')}")

agg = aggregate_records(days=DAYS)
aggregates = agg.get("aggregates") or {}
total = (aggregates.get("totalCount") or {}).get("count", 0)
print(f"\nValidation records in last {DAYS} day(s): {total}")

print("\nBy ruleSetId:")
for bucket in (aggregates.get("ruleSetIds") or {}).get("uniqueValueBuckets") or []:
    print(f"  {bucket.get('value')}: {bucket.get('count')}")

print("\nBy passedValidation:")
for bucket in (aggregates.get("passedBreakdown") or {}).get("uniqueValueBuckets") or []:
    print(f"  {bucket.get('value')}: {bucket.get('count')}")

job_buckets = (aggregates.get("jobRuns") or {}).get("uniqueValueBuckets") or []
print(f"\nDistinct jobRunId values: {len(job_buckets)}")

In [ ]:
def list_validation_records(
    *,
    days: float | None = None,
    max_records: int | object | None = ...,
    passed_validation: bool | None = None,
    rule_set_id: str | None = None,
    print_report: bool = False,
) -> list[dict]:
    """Print and return DataQualityValidationRecord rows from the records stream.

    Defaults: ``DAYS`` / ``MAX_VALIDATION_RECORDS`` from the first cell.
    Set ``MAX_VALIDATION_RECORDS = None`` there for the full window, or pass ``max_records=None`` here.
    """
    resolved_days = DAYS if days is None else days
    resolved_max = MAX_VALIDATION_RECORDS if max_records is ... else max_records
    equals: dict[str, object] = {}
    if passed_validation is not None:
        equals["passedValidation"] = passed_validation
    if rule_set_id:
        equals["ruleSetId"] = rule_set_id

    records = filter_records(
        container=VALIDATION_CONTAINER,
        days=resolved_days,
        max_records=resolved_max,
        equals=equals or None,
    )
    for record in records:
        props = _record_props(record, VALIDATION_CONTAINER)
        focus = props.get("focusNodeInstance") or {}
        focus_label = (
            f"{focus.get('space')}/{focus.get('externalId')}"
            if isinstance(focus, dict) and focus.get("externalId")
            else props.get("focusNode") or "-"
        )
        failed = props.get("failedConstraints") or []
        print(
            f"{record.get('externalId')} | passed={props.get('passedValidation')} | "
            f"ruleSet={props.get('ruleSetId')}:{props.get('ruleSetVersion')} | "
            f"job={props.get('jobRunId')} | focus={focus_label} | failed={failed}"
        )
        if print_report:
            report = props.get("validationReport") or {}
            for violation in report.get("violations") or []:
                print(
                    f"    [{violation.get('severity')}] {violation.get('constraint')}: "
                    f"{violation.get('message')}"
                )
    print(f"{len(records)} validation record(s) (window={resolved_days}d, max={resolved_max})")
    return records


# Sample: latest records. Tune DAYS / MAX_VALIDATION_RECORDS in the first cell.
validation_records = list_validation_records()
print()
_ = list_validation_records(passed_validation=False, print_report=True)

In [ ]:
def list_rule_engine_results(
    *,
    days: float | None = None,
    max_records: int | object | None = ...,
    rule_id: str | None = None,
    rule_set_id: str | None = None,
) -> list[dict]:
    """Print and return RuleEngineResult inference records from the same stream.

    Defaults: ``DAYS`` / ``MAX_RULE_ENGINE_RECORDS`` from the first cell.
    """
    resolved_days = DAYS if days is None else days
    resolved_max = MAX_RULE_ENGINE_RECORDS if max_records is ... else max_records
    equals: dict[str, object] = {}
    if rule_id:
        equals["ruleId"] = rule_id
    if rule_set_id:
        equals["ruleSetId"] = rule_set_id

    # Aggregate scoped to RuleEngineResult (unfiltered totalCount is stream-wide).
    with _with_alpha_headers():
        agg = client.post(
            f"{API_URL}/streams/{RECORDS_STREAM_ID}/records/aggregate",
            json={
                "lastUpdatedTime": {"gt": _ms_ago(resolved_days)},
                "filter": {
                    "equals": {
                        "property": [DATA_QUALITY_SPACE, RULE_ENGINE_CONTAINER, "resultType"],
                        "value": "Inference",
                    }
                },
                "aggregates": {
                    "totalCount": {"count": {}},
                    "ruleIds": {
                        "uniqueValues": {"property": [DATA_QUALITY_SPACE, RULE_ENGINE_CONTAINER, "ruleId"]}
                    },
                },
            },
        ).json()
    total = ((agg.get("aggregates") or {}).get("totalCount") or {}).get("count", 0)
    print(f"RuleEngineResult aggregate count (last {resolved_days}d): {total}")
    for bucket in ((agg.get("aggregates") or {}).get("ruleIds") or {}).get("uniqueValueBuckets") or []:
        print(f"  ruleId={bucket.get('value')}: {bucket.get('count')}")

    if "resultType" not in equals:
        # Always present on RuleEngineResult; scopes the filter to this container.
        equals["resultType"] = "Inference"

    records = filter_records(
        container=RULE_ENGINE_CONTAINER,
        days=resolved_days,
        max_records=resolved_max,
        equals=equals,
    )

    for record in records:
        props = _record_props(record, RULE_ENGINE_CONTAINER)
        focus = props.get("focusNodeInstance") or {}
        focus_label = (
            f"{focus.get('space')}/{focus.get('externalId')}"
            if isinstance(focus, dict) and focus.get("externalId")
            else props.get("focusNode") or "-"
        )
        print(
            f"{record.get('externalId')} | ruleId={props.get('ruleId')} | "
            f"ruleSet={props.get('ruleSetId')}:{props.get('ruleSetVersion')} | "
            f"value={props.get('resultValue')} | focus={focus_label} | run={props.get('runId')}"
        )
    print(f"{len(records)} rule-engine result(s) listed (window={resolved_days}d, max={resolved_max})")
    return records


rule_engine_results = list_rule_engine_results()

## Data products and rule sets

Catalog resources published by Toolkit (`data_products/`, `rulesets/`).

In [ ]:
def list_ruleset_versions(external_id: str) -> list[str]:
    return [str(item["version"]) for item in _list_items(f"{RULESETS_URL}/{external_id}/versions")]


def list_rulesets() -> list[dict]:
    """Print and return all rule sets with versions."""
    rulesets = _list_items(RULESETS_URL)
    for ruleset in rulesets:
        ruleset["versions"] = list_ruleset_versions(ruleset["externalId"])
        print(f"{ruleset['externalId']} | {ruleset.get('name')} | versions: {ruleset['versions']}")
    print(f"{len(rulesets)} rule set(s)")
    return rulesets


def list_data_product_versions(external_id: str) -> list[dict]:
    return _list_items(f"{DATA_PRODUCTS_URL}/{external_id}/versions", limit=10)


def list_data_products() -> list[dict]:
    """Print and return all data products with versions and quality.rules."""
    data_products = _list_items(DATA_PRODUCTS_URL)
    for data_product in data_products:
        data_product["versions"] = list_data_product_versions(data_product["externalId"])
        marker = " ← module" if data_product["externalId"] == DATA_PRODUCT_EXTERNAL_ID else ""
        print(f"{data_product['externalId']} | {data_product.get('name')}{marker}")
        for version in data_product["versions"]:
            rules = [f"{rule['externalId']}:{rule['version']}" for rule in version.get("quality", {}).get("rules", [])]
            views = [
                f"{view.get('space')}/{view.get('externalId')}/{view.get('version')}"
                for view in version.get("views", []) or []
            ]
            print(f"    {version['version']} ({version.get('status')}) | rules: {rules}")
            if views:
                print(f"        views: {views}")
            spaces = version.get("instanceSpaces") or []
            if spaces:
                print(f"        instanceSpaces: {spaces}")
    print(f"{len(data_products)} data product(s)")
    return data_products


rulesets = list_rulesets()
print()
data_products = list_data_products()

## Runtime: function, workflows, triggers, monitors

Resources created by `deploy_infrastructure.py` (not Toolkit).

`inspect_function_modes()` reads recent logs from CDF Function `data-quality-validation`, groups them by `validation_type` (mode), and reports execution problems. If a mode has no execution errors it prints `mode <name> running without error`.

When a mode has problems, set `DUMP_FULL_LOGS = True` in the first code cell (or call `dump_function_call_log(call_id=...)`) to print the full function call log in the notebook.

In [ ]:
def list_dq_function() -> object | None:
    """Print and return the data-quality-validation CDF Function."""
    function = client.functions.retrieve(external_id=FUNCTION_EXTERNAL_ID)
    if function is None:
        print(f"Function '{FUNCTION_EXTERNAL_ID}' not found")
        return None
    print(
        f"{function.external_id} | id={function.id} | name={function.name} | "
        f"status={function.status} | runtime={function.function_path}"
    )
    schedules = [
        schedule
        for schedule in client.functions.schedules.list(limit=None)
        if schedule.function_external_id == FUNCTION_EXTERNAL_ID
        or schedule.function_id == function.id
    ]
    print(f"{len(schedules)} schedule(s) on this function")
    for schedule in schedules:
        print(f"  id={schedule.id} | {schedule.name} | cron={schedule.cron_expression}")
    return function


def list_dq_workflows() -> list:
    """Print DQ-related workflows and their triggers."""
    workflows = [
        workflow
        for workflow in client.workflows.list(limit=None)
        if any(workflow.external_id.startswith(prefix) or workflow.external_id == prefix for prefix in DQ_WORKFLOW_PREFIXES)
    ]
    triggers = list(client.workflows.triggers.list(limit=None))
    triggers_by_workflow: dict[str, list] = {}
    for trigger in triggers:
        triggers_by_workflow.setdefault(trigger.workflow_external_id, []).append(trigger)

    for workflow in sorted(workflows, key=lambda item: item.external_id):
        workflow_triggers = triggers_by_workflow.get(workflow.external_id, [])
        print(f"{workflow.external_id} | triggers={len(workflow_triggers)}")
        for trigger in workflow_triggers:
            print(f"    {trigger.external_id}")
    print(f"{len(workflows)} DQ workflow(s)")
    return workflows


def list_monitor_schedules() -> list:
    """Print function schedules whose name starts with 'Monitor '."""
    schedules = [
        schedule
        for schedule in client.functions.schedules.list(limit=None)
        if (schedule.name or "").startswith("Monitor ")
    ]
    for schedule in schedules:
        print(f"id={schedule.id} | {schedule.name} | cron={schedule.cron_expression}")
    print(f"{len(schedules)} monitor schedule(s)")
    return schedules


def _function_call_log_messages(function_id: int, call_id: int) -> list[str]:
    """Fetch call log lines via REST.

    The SDK ``get_logs`` loader requires every item to have ``message`` and raises
    ``KeyError`` when CDF returns an entry without it.
    """
    response = client.get(f"{API_URL}/functions/{function_id}/calls/{call_id}/logs")
    return [str(item.get("message") or "") for item in response.json().get("items", [])]


def dump_function_call_log(
    call_id: int,
    function_id: int | None = None,
    *,
    max_lines: int | None = None,
) -> list[str]:
    """Print the full log for one ``data-quality-validation`` call.

    Args:
        call_id: Function call id from ``inspect_function_modes`` / ``list_recent_function_calls``.
        function_id: Optional function id (resolved from external id when omitted).
        max_lines: If set, print only the last N lines (useful for huge timeseries runs).
    """
    resolved_function_id = function_id
    if resolved_function_id is None:
        function = client.functions.retrieve(external_id=FUNCTION_EXTERNAL_ID)
        if function is None:
            print(f"Function '{FUNCTION_EXTERNAL_ID}' not found")
            return []
        resolved_function_id = function.id

    messages = _function_call_log_messages(resolved_function_id, call_id)
    to_print = messages[-max_lines:] if max_lines is not None else messages
    print(f"===== full log: function={resolved_function_id} call={call_id} ({len(messages)} line(s)) =====")
    if max_lines is not None and len(messages) > max_lines:
        print(f"(showing last {max_lines} of {len(messages)} lines)")
    for message in to_print:
        print(message)
    print(f"===== end log call={call_id} =====")
    return messages


def _call_mode(messages: list[str]) -> str:
    for message in messages:
        if message.startswith("validation_type:"):
            return message.split(":", 1)[1].strip() or "unknown"
    return "unknown"


def _is_execution_problem(message: str) -> bool:
    """True for function/runtime failures — not SHACL pass/fail on data."""
    if not message or message.startswith("Posted ["):
        return False
    # Noise from the function runtime / SDK version banner.
    if "IndslUserWarning" in message or "disable_pypi_version_check" in message:
        return False
    if message.startswith("ERROR") or "Error posting record" in message:
        return True
    if message.startswith("Traceback") or "Exception:" in message or "CogniteAPIError" in message:
        return True
    if message.startswith("[prefetch]") and "failed" in message.lower():
        return True
    if "could not resolve stream type" in message.lower():
        return True
    lowered = message.lower()
    return " fatal" in f" {lowered}" or lowered.startswith("fatal")


def list_recent_function_calls(limit: int = 15) -> list:
    """Print recent data-quality-validation calls with validation_type from logs."""
    function = client.functions.retrieve(external_id=FUNCTION_EXTERNAL_ID)
    if function is None:
        print(f"Function '{FUNCTION_EXTERNAL_ID}' not found")
        return []
    calls = list(client.functions.calls.list(function_id=function.id, limit=limit))
    for call in calls:
        messages = _function_call_log_messages(function.id, call.id)
        mode = _call_mode(messages)
        posted = sum(1 for msg in messages if "Posted [" in msg and " record for " in msg)
        failed_posts = sum(1 for msg in messages if "Posted [FAILED]" in msg)
        errors = sum(1 for msg in messages if _is_execution_problem(msg))
        print(
            f"call={call.id} | {call.status} | start={call.start_time} | "
            f"mode={mode} | posted_records≈{posted} "
            f"(failed={failed_posts}) | exec_errors={errors}"
        )
    print(f"{len(calls)} recent function call(s)")
    return calls


def inspect_function_modes(
    limit: int = 40,
    max_error_samples: int = 5,
    *,
    dump_full_logs: bool = False,
    max_full_log_calls: int = 2,
    full_log_max_lines: int | None = None,
) -> dict[str, dict]:
    """Group recent function calls by mode and report execution problems.

    Modes come from the ``validation_type:`` log line (e.g. ``timeseries``,
    ``data_product_sync``, ``historic_queue_manager``, ``shacl``, …).

    Args:
        limit: How many recent calls to inspect.
        max_error_samples: Max distinct error lines printed per mode.
        dump_full_logs: When True, print full logs for problem calls.
        max_full_log_calls: Max problem calls to dump per mode (avoids huge output).
        full_log_max_lines: Optional last-N lines cap for each dumped log.

    Returns:
        Per-mode summary dicts with call counts and sampled error lines.
    """
    function = client.functions.retrieve(external_id=FUNCTION_EXTERNAL_ID)
    if function is None:
        print(f"Function '{FUNCTION_EXTERNAL_ID}' not found")
        return {}

    calls = list(client.functions.calls.list(function_id=function.id, limit=limit))
    by_mode: dict[str, dict] = {}
    function_id = function.id

    for call in calls:
        messages = _function_call_log_messages(function_id, call.id)
        mode = _call_mode(messages)
        summary = by_mode.setdefault(
            mode,
            {
                "calls": 0,
                "failed_status": 0,
                "calls_with_errors": 0,
                "error_lines": [],
                "seen_errors": set(),
                "problem_call_ids": [],
                "function_id": function_id,
            },
        )
        summary["calls"] += 1
        status = (call.status or "").lower()
        if status in {"failed", "timeout", "timed_out"}:
            summary["failed_status"] += 1

        problems = [msg for msg in messages if _is_execution_problem(msg)]
        if problems or status in {"failed", "timeout", "timed_out"}:
            summary["calls_with_errors"] += 1
            if call.id not in summary["problem_call_ids"]:
                summary["problem_call_ids"].append(call.id)
        for message in problems:
            # Collapse repeated prefetch / stack lines to unique samples.
            key = message.split(":", 1)[0] if message.startswith("[prefetch]") else message[:160]
            if key in summary["seen_errors"]:
                continue
            summary["seen_errors"].add(key)
            if len(summary["error_lines"]) < max_error_samples:
                summary["error_lines"].append((call.id, message[:300]))

    print(f"Function: {FUNCTION_EXTERNAL_ID} (id={function_id}) — inspected {len(calls)} recent call(s)")
    if dump_full_logs:
        print(f"Full log dump enabled (max {max_full_log_calls} problem call(s) per mode)")
    else:
        print("Full log dump off — set DUMP_FULL_LOGS = True in the first cell, or dump_function_call_log(call_id=...)")
    print()
    if not by_mode:
        print("No recent function calls found.")
        return {}

    for mode in sorted(by_mode):
        summary = by_mode[mode]
        summary.pop("seen_errors", None)
        if summary["calls_with_errors"] == 0 and summary["failed_status"] == 0:
            print(f"mode {mode} running without error ({summary['calls']} call(s))")
            continue
        print(
            f"mode {mode} — PROBLEMS in {summary['calls_with_errors']}/{summary['calls']} call(s)"
            + (f", failed/timeout status={summary['failed_status']}" if summary["failed_status"] else "")
        )
        print(f"  problem call id(s): {summary['problem_call_ids'][:5]}")
        for call_id, line in summary["error_lines"]:
            print(f"  • [call {call_id}] {line}")
        if not summary["error_lines"] and summary["failed_status"]:
            print("  • call status Failed/Timeout — dump the full log for details")
        if dump_full_logs:
            for call_id in summary["problem_call_ids"][:max_full_log_calls]:
                print()
                dump_function_call_log(
                    call_id,
                    function_id=function_id,
                    max_lines=full_log_max_lines,
                )

    return by_mode


dq_function = list_dq_function()
print()
dq_workflows = list_dq_workflows()
print()
monitor_schedules = list_monitor_schedules()
print()
recent_calls = list_recent_function_calls(limit=RECENT_FUNCTION_CALLS_LIMIT)
print()
mode_health = inspect_function_modes(
    limit=MODE_INSPECT_LIMIT,
    max_error_samples=MAX_ERROR_SAMPLES,
    dump_full_logs=DUMP_FULL_LOGS,
    max_full_log_calls=MAX_FULL_LOG_CALLS,
    full_log_max_lines=FULL_LOG_MAX_LINES,
)

# Example after a problem summary:
# dump_function_call_log(call_id=4584231214731569)

## DMS state in dataQualitySpace

Spaces, containers/views, and historic queue / orchestration nodes.

In [ ]:
from cognite.client.data_classes.data_modeling.ids import ViewId
from cognite.client.exceptions import CogniteAPIError


def list_module_spaces() -> list:
    """Print spaces owned/used by this module."""
    wanted = {DATA_QUALITY_SPACE, DATA_PRODUCT_SCHEMA_SPACE}
    spaces = [space for space in client.data_modeling.spaces.list(limit=None) if space.space in wanted]
    for space in spaces:
        print(f"{space.space} | name={space.name} | description={space.description or ''}")
    missing = wanted - {space.space for space in spaces}
    for space_id in sorted(missing):
        if space_id:
            print(f"{space_id} | NOT FOUND")
    print(f"{len(spaces)} module space(s) present")
    return spaces


def list_data_quality_schema() -> None:
    """Print containers and views in dataQualitySpace."""
    containers = client.data_modeling.containers.list(space=DATA_QUALITY_SPACE, limit=None)
    views = client.data_modeling.views.list(space=DATA_QUALITY_SPACE, limit=None)
    print(f"Containers in {DATA_QUALITY_SPACE}:")
    for container in sorted(containers, key=lambda item: item.external_id):
        print(f"  {container.external_id} | usedFor={container.used_for}")
    print(f"{len(containers)} container(s)")
    print(f"\nViews in {DATA_QUALITY_SPACE}:")
    for view in sorted(views, key=lambda item: item.external_id):
        stream = getattr(view, "stream_id", None) or getattr(view, "streamId", None)
        extra = f" | streamId={stream}" if stream else ""
        print(f"  {view.external_id}/{view.version}{extra}")
    print(f"{len(views)} view(s)")


def list_historic_queue_jobs(space: str | None = None) -> list[dict]:
    """Print HistoricJobQueue nodes."""
    resolved_space = space or DATA_QUALITY_SPACE
    view_id = ViewId(space=resolved_space, external_id=HISTORIC_JOB_QUEUE_VIEW, version=HISTORIC_JOB_QUEUE_VIEW_VERSION)
    try:
        nodes = client.data_modeling.instances.list(instance_type="node", sources=[view_id], limit=None)
    except CogniteAPIError as exc:
        print(f"Could not list HistoricJobQueue in space {resolved_space}: {exc}")
        return []

    jobs: list[dict] = []
    for node in nodes:
        props = dict(node.properties.get(view_id) or {}) if node.properties else {}
        job = {"external_id": node.external_id, "space": node.space, **props}
        jobs.append(job)
        print(
            f"{job['external_id']} | status={job.get('status') or '?'} | "
            f"view={job.get('view_external_id') or '?'} | "
            f"dp={job.get('data_product_external_id') or '?'}:{job.get('data_product_version') or '?'} | "
            f"orch={job.get('orchestration_id') or '-'}"
        )
    print(f"{len(jobs)} historic queue job(s)")
    return jobs


module_spaces = list_module_spaces()
print()
list_data_quality_schema()
print()
historic_jobs = list_historic_queue_jobs()

## Tips

- Tunables live in the **first code cell** (`DAYS`, `MAX_VALIDATION_RECORDS`, `DUMP_FULL_LOGS`, …).
- Failed-only: `list_validation_records(passed_validation=False, print_report=True)`.
- One rule set: `list_validation_records(rule_set_id="asset_shacl_rules")`.
- Function modes: `inspect_function_modes()` — execution errors only (not SHACL `Posted [FAILED]` data findings). Set `DUMP_FULL_LOGS = True` or call `dump_function_call_log(call_id=...)` for the full log.
- For delete / cleanup of rule sets, data products, stale workflows, or stuck historic jobs, use [`manage_rulesets.ipynb`](manage_rulesets.ipynb).
- Field reference: [Records output](https://cognite-data-quality-validation.readthedocs-hosted.com/en/latest/usage/records_output.html).